# 📱 スマホだけでセットアップ（Garmin週次レポート）

このノートブックを上から順に実行すると、**PCなし・スマホだけ**で次が完了します。

1. Garmin の認証トークンを生成
2. GitHub にリポジトリを作成し、必要ファイルを自動アップロード
3. （最後だけ手動）GitHub に秘密情報を登録 → 週次自動実行を開始

---
### 事前に用意するもの（すべてスマホのブラウザで取得できます）
- **Google アカウント**（このColabを動かすため）
- **Garmin Connect** のメール・パスワード
- **Gemini APIキー** … `aistudio.google.com` →「Get API key」
- **LINE** … `developers.line.biz` でチャネルアクセストークンと自分のユーザーID
- **GitHub アカウント** と **個人アクセストークン(PAT)**
  - `github.com` → Settings → Developer settings → Personal access tokens →
    **Tokens (classic)** → Generate new token → スコープ **repo** と **workflow** にチェック

> 各キーの詳しい取得手順は、アップロードされる `README.md` の第2章にもあります。

▶ セル左の実行ボタンを上から順に押してください。


## 手順1: 必要ライブラリのインストール

In [ ]:
!pip install garminconnect requests -q
print('✅ インストール完了')

## 手順2: Garmin トークンを生成

Garmin のメール・パスワード（2段階認証ありの場合はコードも）を入力します。
成功すると長いトークン文字列が `GARMIN_TOKENS` 変数に入ります（手順3で自動利用）。

> 補足: ColabはGoogleのデータセンターから動くため、ごく稀にGarminのログインが
> 弾かれることがあります。その場合は時間をおいて再実行してください。


In [ ]:
import getpass
from garminconnect import Garmin

email = input("Garmin email: ").strip()
password = getpass.getpass("Garmin password: ")
g = Garmin(email, password, prompt_mfa=lambda: input("MFAコード（求められたら入力）: ").strip())
g.login()
GARMIN_TOKENS = g.client.dumps()
print("✅ Garminログイン成功。トークンを取得しました（長さ:", len(GARMIN_TOKENS), "文字）")


## 手順3: GitHub リポジトリを作成し、ファイルを自動アップロード

GitHub の **個人アクセストークン(PAT)** と、作成したい **リポジトリ名** を入力します。
リポジトリ（private）が作られ、必要なファイル一式が自動で配置されます。


In [ ]:
PROJECT_FILES_B64 = {
    'garmin_weekly_report.py': '...',
    'requirements.txt': 'IyBHYXJtaW4gQ29ubmVjdCDpgLHmrKHjg6njg7Pjg4vjg7PjgrDjg6zjg53jg7zjg4gg5L6d5a2Y44OR44OD44Kx44O844K4CiMgICDjgqTjg7Pjgrnjg4jjg7zjg6s6IHBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy50eHQKCiMgR2FybWluIENvbm5lY3Qg6Z2e5YWs5byPQVBJ44Op44OD44OR44O877yIMC4zLjDku6XpmY3jga/jg43jgqTjg4bjgqPjg5boqo3oqLzvvIkKZ2FybWluY29ubmVjdD49MC4zLjAKCiMgSFRUUOODquOCr+OCqOOCueODiO+8iEdlbWluaSBBUEkgLyBMSU5FIEFQSSDlkbzjgbPlh7rjgZfnlKjvvIkKcmVxdWVzdHM+PTIuMzEuMAoKIyDku6XkuIvjga8gZ2FybWluY29ubmVjdCDjgYzoh6rli5XjgaflhaXjgozjgovkvp3lrZjvvIjmmI7npLrjgZfjgabjgYrjgY/jgajnkrDlooPlt67nlbDjgavlvLfjgYTvvIkKY3VybF9jZmZpCnVhLWdlbmVyYXRvcgo=',
    'config.example.ini': 'IyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyAg6Kit5a6a44OV44Kh44Kk44Or77yI44GT44Gu44OV44Kh44Kk44Or44KSIGNvbmZpZy5pbmkg44Go44GE44GG5ZCN5YmN44Gn44Kz44OU44O844GX44Gm5L2/44GE44G+44GZ77yJCiMKIyAgIGNwIGNvbmZpZy5leGFtcGxlLmluaSBjb25maWcuaW5pICAg77yITWFjIC8gTGludXjvvIkKIyAgIGNvcHkgY29uZmlnLmV4YW1wbGUuaW5pIGNvbmZpZy5pbmkg77yIV2luZG93c++8iQojCiMgIOWQhOWApOOBruWPluW+l+aWueazleOBryBSRUFETUUubWQg44KS5Y+C54Wn44GX44Gm44GP44Gg44GV44GE44CCCiMgIGNvbmZpZy5pbmkg44Gr44Gv44OR44K544Ov44O844OJ44O7QVBJ44Kt44O844GM5ZCr44G+44KM44G+44GZ44CC57W25a++44Gr5LuW5Lq644Gr5rih44GX44Gf44KKCiMgIEdpdEh1YuetieOBq+WFrOmWi+OBl+OBn+OCiuOBl+OBquOBhOOBp+OBj+OBoOOBleOBhO+8iC5naXRpZ25vcmUg44Gn6Zmk5aSW5riI44G/77yJ44CCCiMKIyAg4oC7IEdpdEh1YiBBY3Rpb25z562J44Gu44Kv44Op44Km44OJ5a6f6KGM44Gn44GvIGNvbmZpZy5pbmkg44Gv5LiN6KaB44Gn44GZ44CCCiMgICAg5ZCM44GY6aCF55uu44KS55Kw5aKD5aSJ5pWw77yIU2VjcmV0c++8ieOBp+a4oeOBm+OBvuOBme+8iOeSsOWig+WkieaVsOOBjOWEquWFiOOBleOCjOOBvuOBme+8ieOAggojICAgIOWvvuW/nOOBmeOCi+eSsOWig+WkieaVsOWQjeOBryBSRUFETUUubWQg44Gu44CMUEPjg6zjgrnpgYvnlKjjgI3jgpLlj4LnhafjgIIKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKCltnYXJtaW5dCiMgR2FybWluIENvbm5lY3Qg44Gu44Ot44Kw44Kk44Oz5oOF5aCxCmVtYWlsICAgID0geW91cl9lbWFpbEBleGFtcGxlLmNvbQpwYXNzd29yZCA9IHlvdXJfcGFzc3dvcmQKCltnZW1pbmldCiMgR29vZ2xlIEFJIFN0dWRpbyDjgaflj5blvpfjgZfjgZ9BUEnjgq3jg7zvvIjjgq/jg6zjgqvkuI3opoHjg7vnhKHmlpnmnqDvvIkKYXBpX2tleSA9IHlvdXJfZ2VtaW5pX2FwaV9rZXkKIyDkvb/nlKjjg6Ljg4fjg6vvvIjnhKHmlpnmnqA6IGdlbWluaS0yLjUtZmxhc2ggLyDjgZXjgonjgavou73ph486IGdlbWluaS0yLjUtZmxhc2gtbGl0Ze+8iQptb2RlbCAgID0gZ2VtaW5pLTIuNS1mbGFzaAoKW2xpbmVdCiMgTElORSBNZXNzYWdpbmcgQVBJIOOBruODgeODo+ODjeODq+OCouOCr+OCu+OCueODiOODvOOCr+ODs++8iOmVt+acn++8ieOBqOOAgeiHquWIhuOBruODpuODvOOCtuODvElECmNoYW5uZWxfYWNjZXNzX3Rva2VuID0geW91cl9saW5lX2NoYW5uZWxfYWNjZXNzX3Rva2VuCnVzZXJfaWQgICAgICAgICAgICAgID0geW91cl9saW5lX3VzZXJfaWQKCltnb2FsXQojIOebruaomeOBqOODl+ODreODleOCo+ODvOODq++8iOiHqueUseOBq+abuOOBjeaPm+OBiOOBpuOBj+OBoOOBleOBhOOAgkxMTeOBruWIhuaekOOBq+a4oeOBleOCjOOBvuOBme+8iQptYXJhdGhvbl90aW1lID0gM+aZgumWkzMw5YiGCnJhY2VfcGFjZSAgICAgPSA0OjU4L2ttCiMg6KSH5pWw6KGM44Gn5pu444GP5aC05ZCI44Gv44CBMuihjOebruS7pemZjeOCkuW/heOBmuWNiuinkuOCueODmuODvOOCueOBp+Wtl+S4i+OBkuOBl+OBpuOBj+OBoOOBleOBhApwcm9maWxlID0KICAgIC0g55uu5qiZOiDjg5Xjg6vjg57jg6njgr3jg7Mg44K144OWMy4177yIM+aZgumWkzMw5YiG5YiH44KK77yJCiAgICAtIOW/heimgeODrOODvOOCueODmuODvOOCuTogNDo1OC9rbQogICAgLSDjg4jjg6zjg7zjg4vjg7PjgrDmlrnph506IDgwLzIw44Or44O844Or77yI5pyJ6YW457SgODAl44CB6auY5by35bqmMjAl77yJCiAgICAtIOODreODs+OCsOi1sOebruaomTog6YCxMeWbniAyMOOAnDMwa20KICAgIC0g6YCx6ZaT6LWw6KGM6Led6Zui55uu5qiZOiA1MOOAnDcwa20KICAgIC0g44Os44O844K55pelOiAyMDI25bm0MTHmnIgz5pel44CA576k6aas44Oe44Op44K944OzCg==',
    'README.md': '...',
    'SMARTPHONE_SETUP.md': 'IyDwn5OxIOOCueODnuODm+OBoOOBkeOBp+OCu+ODg+ODiOOCouODg+ODl+OBmeOCi+aJi+mghgoKUEPjgYzjgarjgY/jgabjgoLjgIEqKuOCueODnuODm+OBruODluODqeOCpuOCtuOBoOOBkSoq44Gn44K744OD44OI44Ki44OD44OX44Gn44GN44G+44GZ44CCCumbo+aJgOOBruOAjEdpdEh1YuOBuOOBruODleOCoeOCpOODq+mFjee9ruOAjeOBr+S7mOWxnuOBriBDb2xhYiDjg47jg7zjg4jjg5bjg4Pjgq8K77yIYHNldHVwX2Zyb21fcGhvbmUuaXB5bmJg77yJ44GM6Ieq5YuV44Gn44KE44Gj44Gm44GP44KM44G+44GZ44CCCgotLS0KCiMjIOWFqOS9k+OBrua1geOCjAoKYGBgCuKRoCDlkITnqK7jgq3jg7zjgpLjgrnjg57jg5vjgaflj5blvpfvvIhHZW1pbmkgLyBMSU5FIC8gR2l0SHViIFBBVO+8iQogICAgICAgIOKGkwrikaEgQ29sYWLjgafjg47jg7zjg4jjg5bjg4Pjgq/jgpLlrp/ooYzvvIhHYXJtaW7jg4jjg7zjgq/jg7PnlJ/miJAg4oaSIEdpdEh1YuOBuOiHquWLlemFjee9ru+8iQogICAgICAgIOKGkwrikaIgR2l0SHVi44Gn56eY5a+G5oOF5aCx44KS55m76YyyIOKGkiDoh6rli5Xlrp/ooYzjgrnjgr/jg7zjg4gKYGBgCgrmiYDopoHmmYLplpPjga/jgYrjgojjgZ0yMOOAnDMw5YiG44CC44OI44O844Kv44Oz6aGe44Gu5Y+W5b6X44GM5Yid44KB44Gm44Gg44Go5bCR44GX5pmC6ZaT44GM44GL44GL44KK44G+44GZ44CCCgotLS0KCiMjIOKRoCDkuovliY3jgavlj5blvpfjgZnjgovjgoLjga7vvIjjgZnjgbnjgabjgrnjg57jg5vjgafvvIkKCiMjIyBHZW1pbmkgQVBJ44Kt44O8CjEuIOODluODqeOCpuOCtuOBpyBgYWlzdHVkaW8uZ29vZ2xlLmNvbWAg44KS6ZaL44GNR29vZ2xl44Gn44Ot44Kw44Kk44OzCjIuIOOAjEdldCBBUEkga2V544CN4oaS44CMQ3JlYXRlIEFQSSBrZXnjgI3ihpIg6KGo56S644GV44KM44Gf5paH5a2X5YiX44KS44Kz44OU44O877yI44Oh44Oi44Ki44OX44Oq44Gr5L+d5a2Y77yJCgojIyMgTElORe+8iOODgeODo+ODjeODq+OCouOCr+OCu+OCueODiOODvOOCr+ODs++8i+ODpuODvOOCtuODvElE77yJCjEuIGBkZXZlbG9wZXJzLmxpbmUuYml6L2NvbnNvbGUvYCDjgatMSU5F44Gn44Ot44Kw44Kk44OzCjIuIOODl+ODreODkOOCpOODgOODvOS9nOaIkCDihpLjgIzmlrDopo/jg4Hjg6Pjg43jg6vkvZzmiJDjgI3ihpLjgIxNZXNzYWdpbmcgQVBJ44CNCjMuIOOAjE1lc3NhZ2luZyBBUEnoqK3lrprjgI3jgr/jg5Yg4oaS44CM44OB44Oj44ON44Or44Ki44Kv44K744K544OI44O844Kv44Oz77yI6ZW35pyf77yJ44CN44KS55m66KGM44GX44Gm44Kz44OU44O8CjQuIOOAjOODgeODo+ODjeODq+WfuuacrOioreWumuOAjeOCv+ODliDihpLjgIzjgYLjgarjgZ/jga7jg6bjg7zjgrbjg7xJROOAje+8iFXlp4vjgb7jgorvvInjgpLjgrPjg5Tjg7wKNS4gQm9044KSKirlj4vjgaDjgaHov73liqAqKu+8iOWQjOOCv+ODluOBruWPi+OBoOOBoei/veWKoOODquODs+OCr++8j0JvdCBJROaknOe0ouOBi+OCieOAguOBk+OCjOOCkuOBl+OBquOBhOOBqOWxiuOBjeOBvuOBm+OCk++8iQoKPiDigLsg55S76Z2i5pON5L2c44Gv44K544Oe44Ob44Gg44Go44KE44KE56qu5bGI44Gn44GZ44CC44OW44Op44Km44K244KS44CMUEPniYjjgrXjgqTjg4jjgpLooajnpLrjgI3jgavjgZnjgovjgajmk43kvZzjgZfjgoTjgZnjgY/jgarjgorjgb7jgZnjgIIKCiMjIyBHaXRIdWIg44Ki44Kr44Km44Oz44OI44Go5YCL5Lq644Ki44Kv44K744K544OI44O844Kv44OzKFBBVCkKMS4gYGdpdGh1Yi5jb21gIOOBp+OCouOCq+OCpuODs+ODiOS9nOaIkO+8j+ODreOCsOOCpOODswoyLiBTZXR0aW5ncyDihpIgRGV2ZWxvcGVyIHNldHRpbmdzIOKGkiBQZXJzb25hbCBhY2Nlc3MgdG9rZW5zIOKGkgogICAqKlRva2VucyAoY2xhc3NpYykqKiDihpLjgIxHZW5lcmF0ZSBuZXcgdG9rZW4gKGNsYXNzaWMp44CNCjMuIOOCueOCs+ODvOODlyAqKnJlcG8qKiDjgaggKip3b3JrZmxvdyoqIOOBq+ODgeOCp+ODg+OCr+OBl+OBpueZuuihjCDihpIg5paH5a2X5YiX44KS44Kz44OU44O8CiAgIO+8iOOBk+OBrueUu+mdouOCkumbouOCjOOCi+OBqOS6jOW6puOBqOihqOekuuOBleOCjOOBquOBhOOBruOBp+W/heOBmuS/neWtmO+8iQoKPiDlkITjgq3jg7zjga7oqbPntLDjga8gYFJFQURNRS5tZGAg56ysMueroOOCguWPgueFp+OAggoKLS0tCgojIyDikaEgQ29sYWIg44Gn44OO44O844OI44OW44OD44Kv44KS5a6f6KGMCgoxLiDjg5bjg6njgqbjgrbjgacgYGNvbGFiLnJlc2VhcmNoLmdvb2dsZS5jb21gIOOCkumWi+OBjUdvb2dsZeOBp+ODreOCsOOCpOODswoyLiDjgIzjg47jg7zjg4jjg5bjg4Pjgq/jgpLplovjgY/jgI3ihpLjgIzjgqLjg4Pjg5fjg63jg7zjg4njgI3ihpIgYHNldHVwX2Zyb21fcGhvbmUuaXB5bmJgIOOCkumBuOaKngogICDvvIjjg5XjgqHjgqTjg6vjga/jgZPjga7jg5Hjg4PjgrHjg7zjgrjlhoXjgILjgrnjg57jg5vjga7jgIzjg5XjgqHjgqTjg6vjgI3jgqLjg5fjg6rnrYnjgYvjgonpgbjjgbnjgb7jgZnvvIkKMy4g5LiK44Gu44K744Or44GL44KJ6aCG44Gr5YaN55Sf44Oc44K/44Oz44KS5oq844GX44Gm5a6f6KGMOgogICAtICoq5omL6aCGMSoqOiDjg6njgqTjg5bjg6njg6rjga7jgqTjg7Pjgrnjg4jjg7zjg6sKICAgLSAqKuaJi+mghjIqKjogR2FybWlu44Gu44Oh44O844Or44O744OR44K544Ov44O844OJ77yI5b+F6KaB44Gq44KJTUZB44Kz44O844OJ77yJ44KS5YWl5YqbIOKGkiDjg4jjg7zjgq/jg7PnlJ/miJAKICAgLSAqKuaJi+mghjMqKjogR2l0SHVi44GuUEFU44Go44Oq44Od44K444OI44Oq5ZCN44KS5YWl5YqbIOKGkiDjg6rjg53jgrjjg4jjg6rkvZzmiJDvvIbjg5XjgqHjgqTjg6voh6rli5XjgqLjg4Pjg5fjg63jg7zjg4kKCuWun+ihjOW+jOOAgeS9nOaIkOOBleOCjOOBn+ODquODneOCuOODiOODquOBrlVSTOOBjOihqOekuuOBleOCjOOBvuOBmeOAggoKPiBDb2xhYuOBr+ODh+ODvOOCv+OCu+ODs+OCv+ODvOOBi+OCieWLleOBj+OBn+OCgeOAgeOBlOOBj+eogOOBq+aJi+mghjLjga5HYXJtaW7jg63jgrDjgqTjg7PjgYzlvL7jgYvjgozjgb7jgZnjgIIKPiDjgZ3jga7loLTlkIjjga/mlbDliIbjgYrjgYTjgablho3lrp/ooYzjgZfjgabjgY/jgaDjgZXjgYTjgILjgZ3jgozjgafjgoLjg4Djg6HjgarjgonjgIxQQ+OCkuS4gOW6puOBoOOBkeWAn+OCiuOCi+OAjeOBruOBjOeiuuWun+OBp+OBmeOAggoKLS0tCgojIyDikaIg56eY5a+G5oOF5aCx44KS55m76Yyy44GX44Gm6Ieq5YuV5a6f6KGM44K544K/44O844OICgrjg5XjgqHjgqTjg6vphY3nva7jgb7jgafjga/oh6rli5XjgafntYLjgo/jgaPjgabjgYTjgb7jgZnjgILmnIDlvozjgavnp5jlr4bmg4XloLHjgaDjgZHmiYvli5XjgafnmbvpjLLjgZfjgb7jgZnvvIjlronlhajjga7jgZ/jgoHvvInjgIIKCjEuIOS9nOaIkOOBleOCjOOBn+ODquODneOCuOODiOODquOCkumWi+OBjwoyLiAqKlNldHRpbmdzIOKGkiBTZWNyZXRzIGFuZCB2YXJpYWJsZXMg4oaSIEFjdGlvbnMqKgozLiAqKlNlY3JldHMqKiDjgr/jg5bjgafmrKHjga4044Gk44KS55m76Yyy77yI44CMTmV3IHJlcG9zaXRvcnkgc2VjcmV044CN77yJOgoKICAgfCBOYW1lIHwgVmFsdWUgfAogICB8LS0tLS0tfC0tLS0tLS18CiAgIHwgYEdBUk1JTl9UT0tFTlNgIHwg5omL6aCGMuOBp+eUn+aIkO+8iOODjuODvOODiOODluODg+OCr+acgOW+jOOBruOCu+ODq+OBp+WGjeihqOekuuOBp+OBjeOBvuOBme+8iSB8CiAgIHwgYEdFTUlOSV9BUElfS0VZYCB8IEdlbWluaSBBUEnjgq3jg7wgfAogICB8IGBMSU5FX0NIQU5ORUxfQUNDRVNTX1RPS0VOYCB8IExJTkXjg4Hjg6Pjg43jg6vjgqLjgq/jgrvjgrnjg4jjg7zjgq/jg7MgfAogICB8IGBMSU5FX1VTRVJfSURgIHwgTElOReODpuODvOOCtuODvElEIHwKCjQuIO+8iOS7u+aEj++8iSoqVmFyaWFibGVzKiog44K/44OW44Gn55uu5qiZ44KS55m76YyyOgogICBgR09BTF9NQVJBVEhPTl9USU1FYCAvIGBHT0FMX1JBQ0VfUEFDRWAgLyBgUlVOTkVSX1BST0ZJTEVgIC8gYEdFTUlOSV9NT0RFTGAKNS4gKipBY3Rpb25zKiog44K/44OW44Gn44Ov44O844Kv44OV44Ot44O844KS5pyJ5Yq55YyWCjYuIOOAjFdlZWtseSBSdW5uaW5nIFJlcG9ydOOAjeKGkuOAjCoqUnVuIHdvcmtmbG93KirjgI3jgafmiYvli5Xlrp/ooYwg4oaSIExJTkXjgavlsYrjgZHjgbDlrozkuoYg8J+OiQoK5Lul6ZmN44Gv5q+O6YCx5pyI5pucIDc6MDDvvIjml6XmnKzmmYLplpPvvInjgavoh6rli5Xjgafjg6zjg53jg7zjg4jjgYzlsYrjgY3jgb7jgZnjgIIKCi0tLQoKIyMg5omL5YuV44Gn44KE44KL5aC05ZCI77yI44OO44O844OI44OW44OD44Kv44KS5L2/44KP44Gq44GE5pa55rOV77yJCgpDb2xhYuOCkuS9v+OCj+OBmkdpdEh1YuOBruODluODqeOCpuOCtuaTjeS9nOOBoOOBkeOBp+OCguWPr+iDveOBp+OBme+8iOOChOOChOaJi+mWk++8ieOAggoKMS4gR2l0SHVi44Gn44Oq44Od44K444OI44Oq44KS5paw6KaP5L2c5oiQCjIuIOOAjEFkZCBmaWxl44CN4oaS44CMQ3JlYXRlIG5ldyBmaWxl44CN44Gn44CB44OV44Kh44Kk44Or5ZCN44GrCiAgIGAuZ2l0aHViL3dvcmtmbG93cy93ZWVrbHktcmVwb3J0LnltbGAg44Gq44Gp44OR44K544KS55u05o6l5YWl5Yqb44GX44CB5Lit6Lqr44KS6LK844KK5LuY44GR44Gm44Kz44Of44OD44OICiAgIO+8iOOBk+OCjOOCkuWQhOODleOCoeOCpOODq+WIhuOBj+OCiui/lOOBmeOAgmAuZ2l0aHViL2Ag44KS5ZCr44KA44OR44K544KC44GT44Gu5pa55rOV44Gq44KJ5L2c44KM44G+44GZ77yJCjMuIEdhcm1pbuODiOODvOOCr+ODs+OBoOOBkeOBr+WIpemAlOeUn+aIkOOBjOW/heimgSDihpIgYFJFQURNRS5tZGAg44Gu44CM44K544Oe44Ob44Gn44OI44O844Kv44Oz55Sf5oiQ44CN44G+44Gf44GvCiAgIOS7mOWxnuODjuODvOODiOODluODg+OCr+OBruaJi+mghjLjga7jgrvjg6vjgaDjgZHlrp/ooYwKNC4g44GC44Go44Gv5LiK6KiY4pGi44Go5ZCM44GYCgotLS0KCiMjIOODiOODqeODluODq+aZggoKfCDnl4fnirYgfCDlr77lh6YgfAp8LS0tLS0tfC0tLS0tLXwKfCDmiYvpoIYy44GnR2FybWlu44Ot44Kw44Kk44Oz5aSx5pWXIHwg5pWw5YiG5b6F44Gj44Gm5YaN5a6f6KGM44CC57aa44GP5aC05ZCI44GvUEPjgpLkuIDluqbjgaDjgZHlgJ/jgorjgaYgYHNldHVwX2dhcm1pbl90b2tlbi5weWAgfAp8IOaJi+mghjPjgacgNDAxLzQwMyB8IFBBVOOBruOCueOCs+ODvOODl+OBqyAqKnJlcG8qKiDjgaggKip3b3JrZmxvdyoqIOOBjOWFpeOBo+OBpuOBhOOCi+OBi+eiuuiqjSB8Cnwg5omL6aCGM+OBpyB3b3JrZmxvdyDjg5XjgqHjgqTjg6vjgaDjgZHlpLHmlZcgfCBQQVTjgasgKip3b3JrZmxvdyoqIOOCueOCs+ODvOODl+OBjOW/heimgSB8CnwgTElOReOBq+WxiuOBi+OBquOBhCB8IEJvdOOCkuWPi+OBoOOBoei/veWKoOOBl+OBn+OBi++8j2BMSU5FX1VTRVJfSURgIOOBjOato+OBl+OBhOOBiyB8CnwgQWN0aW9uc+OBjOWLleOBi+OBquOBhCB8IOODquODneOCuOODiOODquOBrkFjdGlvbnPjgpLmnInlirnljJbjgZfjgZ/jgYvvvI/jg4fjg5Xjgqnjg6vjg4jjg5bjg6njg7Pjg4HjgYsgfAo=',
    '.gitignore': 'IyDoqo3oqLzmg4XloLHvvIjntbblr77jgavjgrPjg5/jg4Pjg4jjgZfjgarjgYTvvIkKY29uZmlnLmluaQoKIyDlrp/ooYzjg63jgrAKZ2FybWluX3JlcG9ydC5sb2cKCiMgR2FybWluIOiqjeiovOODiOODvOOCr+ODs+OBruOCreODo+ODg+OCt+ODpQouZ2FybWluY29ubmVjdC8KZ2FybWluX3Rva2Vucy5qc29uCgojIFB5dGhvbgpfX3B5Y2FjaGVfXy8KKi5weWMKLnZlbnYvCnZlbnYvCmVudi8KCiMgT1MKLkRTX1N0b3JlClRodW1icy5kYgo=',
    'setup_garmin_token.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKR2FybWluIOODiOODvOOCr+ODs+eUn+aIkOODhOODvOODq++8iOacgOWIneOBqzHlm57jgaDjgZHjgIHoh6rliIbjga5QQ+OBp+Wun+ihjO+8iQoKR2l0SHViIEFjdGlvbnMg44Gq44Gp44Gu44Kv44Op44Km44OJ44Gn5q+O5Zue44OR44K544Ov44O844OJ44Ot44Kw44Kk44Oz44GZ44KL44Go44CBCkdhcm1pbiDjgavjg5zjg4Pjg4jmibHjgYTjgZXjgozjgabjg5bjg63jg4Pjgq/jgZXjgozjgoTjgZnjgY/jgarjgorjgb7jgZnjgIIK44GT44Gu44K544Kv44Oq44OX44OI44Gn5LiA5bqm44Gg44GR44Ot44Kw44Kk44Oz44GX44Gm44OI44O844Kv44Oz5paH5a2X5YiX44KS55Sf5oiQ44GX44CBCuOBneOCjOOCkiBHaXRIdWIgU2VjcmV0IGBHQVJNSU5fVE9LRU5TYCDjgavnmbvpjLLjgZnjgovjgajjgIEK44Kv44Op44Km44OJ5YG044Gv44OR44K544Ov44O844OJ54Sh44GX44O744OI44O844Kv44Oz44Gg44GR44Gn44Ot44Kw44Kk44Oz44Gn44GN44G+44GZ44CCCgrkvb/jgYTmlrk6CiAgICBweXRob24gc2V0dXBfZ2FybWluX3Rva2VuLnB5CiAgICDvvIjjg6Hjg7zjg6vjg7vjg5Hjgrnjg6/jg7zjg4njgIHlv4XopoHjgarjgolNRkHjgrPjg7zjg4njgpLlhaXlipvvvIkKICAgIOKGkiDooajnpLrjgZXjgozjgZ/plbfjgYTmloflrZfliJfjgpLjgZnjgbnjgabjgrPjg5Tjg7zjgZfjgaYgU2VjcmV0IGBHQVJNSU5fVE9LRU5TYCDjgavosrzjgorku5jjgZEKCuODiOODvOOCr+ODs+OBruacieWKueacn+mZkOOBr+e0hDHlubTjgafjgZnjgILmnJ/pmZDliIfjgozjgoTjg63jgrDjgqTjg7PlpLHmlZfjgYzlh7rjgZ/jgonlho3lrp/ooYzjgZfjgabjgY/jgaDjgZXjgYTjgIIKIiIiCgppbXBvcnQgZ2V0cGFzcwppbXBvcnQgc3lzCgoKZGVmIG1haW4oKToKICAgIHRyeToKICAgICAgICBmcm9tIGdhcm1pbmNvbm5lY3QgaW1wb3J0IEdhcm1pbgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHN5cy5leGl0KCJnYXJtaW5jb25uZWN0IOOBjOacquOCpOODs+OCueODiOODvOODq+OBp+OBmeOAguWFiOOBqyBgcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLnR4dGAg44KS5a6f6KGM44GX44Gm44GP44Gg44GV44GE44CCIikKCiAgICBwcmludCgiPT09IEdhcm1pbiDjg4jjg7zjgq/jg7PnlJ/miJAgPT09IikKICAgIGVtYWlsID0gaW5wdXQoIkdhcm1pbiBlbWFpbDogIikuc3RyaXAoKQogICAgcGFzc3dvcmQgPSBnZXRwYXNzLmdldHBhc3MoIkdhcm1pbiBwYXNzd29yZDogIikKCiAgICAjIE1GQe+8iDLmrrXpmo7oqo3oqLzvvInjgYzmnInlirnjgarloLTlkIjjga/jgrPjg7zjg4nlhaXlipvjgpLkv4PjgZkKICAgIGdhcm1pbiA9IEdhcm1pbihlbWFpbCwgcGFzc3dvcmQsIHByb21wdF9tZmE9bGFtYmRhOiBpbnB1dCgiTUZB44Kz44O844OJ44KS5YWl5YqbOiAiKS5zdHJpcCgpKQoKICAgIHByaW50KCJcbuODreOCsOOCpOODs+S4rS4uLiIpCiAgICB0cnk6CiAgICAgICAgZ2FybWluLmxvZ2luKCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBzeXMuZXhpdChmIuKdjCDjg63jgrDjgqTjg7PjgavlpLHmlZfjgZfjgb7jgZfjgZ86IHtlfSIpCgogICAgdG9rZW5fc3RyID0gZ2FybWluLmNsaWVudC5kdW1wcygpCgogICAgcHJpbnQoIlxu4pyFIOODreOCsOOCpOODs+aIkOWKn+OAguS4i+OBrjHooYzvvIjplbfjgYTmloflrZfliJfvvInjgpLjgZnjgbnjgabjgrPjg5Tjg7zjgZfjgabjgIEiKQogICAgcHJpbnQoIiAgIEdpdEh1YiBTZWNyZXTjgI5HQVJNSU5fVE9LRU5T44CP44Gr6LK844KK5LuY44GR44Gm44GP44Gg44GV44GE44CCIikKICAgIHByaW50KCIgICDvvIjjg63jg7zjgqvjg6vjgafjg4jjg7zjgq/jg7PpgYvnlKjjgZfjgZ/jgYTloLTlkIjjga/nkrDlooPlpInmlbAgR0FSTUlOX1RPS0VOUyDjgavoqK3lrprvvIlcbiIpCiAgICBwcmludCgi4pSAIiAqIDYwKQogICAgcHJpbnQodG9rZW5fc3RyKQogICAgcHJpbnQoIuKUgCIgKiA2MCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    '.github/workflows/weekly-report.yml': 'bmFtZTogV2Vla2x5IFJ1bm5pbmcgUmVwb3J0CgojIOWun+ihjOOCv+OCpOODn+ODs+OCsApvbjoKICBzY2hlZHVsZToKICAgICMgR2l0SHViIEFjdGlvbnMg44GuIGNyb24g44GvIFVUQyDln7rmupbjgafjgZnjgIIKICAgICMg5pel5pys5pmC6ZaT77yISlNUPVVUQys577yJ5pyI5pucIDA3OjAwIOKGkiBVVEMg5pel5pucIDIyOjAwCiAgICAtIGNyb246ICIwIDIyICogKiAwIgogIHdvcmtmbG93X2Rpc3BhdGNoOiAgICMg5omL5YuV5a6f6KGM44Oc44K/44Oz77yIQWN0aW9uc+OCv+ODluOBi+OCieS7iuOBmeOBkOWun+ihjOOBp+OBjeOCi++8j+WLleS9nOeiuuiqjeeUqO+8iQoKam9iczoKICByZXBvcnQ6CiAgICBydW5zLW9uOiB1YnVudHUtbGF0ZXN0CiAgICBzdGVwczoKICAgICAgLSBuYW1lOiDjg6rjg53jgrjjg4jjg6rjgpLlj5blvpcKICAgICAgICB1c2VzOiBhY3Rpb25zL2NoZWNrb3V0QHY0CgogICAgICAtIG5hbWU6IFB5dGhvbiDjgpLjgrvjg4Pjg4jjgqLjg4Pjg5cKICAgICAgICB1c2VzOiBhY3Rpb25zL3NldHVwLXB5dGhvbkB2NQogICAgICAgIHdpdGg6CiAgICAgICAgICBweXRob24tdmVyc2lvbjogIjMuMTIiCiAgICAgICAgICBjYWNoZTogInBpcCIKCiAgICAgIC0gbmFtZTog5L6d5a2Y44OR44OD44Kx44O844K444KS44Kk44Oz44K544OI44O844OrCiAgICAgICAgcnVuOiBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMudHh0CgogICAgICAtIG5hbWU6IOODrOODneODvOODiOWun+ihjAogICAgICAgIGVudjoKICAgICAgICAgICMg4pSA4pSAIFNlY3JldHPvvIhTZXR0aW5ncyA+IFNlY3JldHMgYW5kIHZhcmlhYmxlcyA+IEFjdGlvbnMgPiBTZWNyZXRz77yJ4pSA4pSACiAgICAgICAgICBHQVJNSU5fVE9LRU5TOiAgICAgICAgICAgICAke3sgc2VjcmV0cy5HQVJNSU5fVE9LRU5TIH19CiAgICAgICAgICBHQVJNSU5fRU1BSUw6ICAgICAgICAgICAgICAke3sgc2VjcmV0cy5HQVJNSU5fRU1BSUwgfX0KICAgICAgICAgIEdBUk1JTl9QQVNTV09SRDogICAgICAgICAgICR7eyBzZWNyZXRzLkdBUk1JTl9QQVNTV09SRCB9fQogICAgICAgICAgR0VNSU5JX0FQSV9LRVk6ICAgICAgICAgICAgJHt7IHNlY3JldHMuR0VNSU5JX0FQSV9LRVkgfX0KICAgICAgICAgIExJTkVfQ0hBTk5FTF9BQ0NFU1NfVE9LRU46ICR7eyBzZWNyZXRzLkxJTkVfQ0hBTk5FTF9BQ0NFU1NfVE9LRU4gfX0KICAgICAgICAgIExJTkVfVVNFUl9JRDogICAgICAgICAgICAgICR7eyBzZWNyZXRzLkxJTkVfVVNFUl9JRCB9fQogICAgICAgICAgIyDilIDilIAgVmFyaWFibGVz77yI5Lu75oSP44CC5ZCM55S76Z2i44GuIFZhcmlhYmxlcyDjgr/jg5bjgILnp5jlr4bjgafjgarjgYToqK3lrprvvInilIDilIAKICAgICAgICAgIEdFTUlOSV9NT0RFTDogICAgICAgICAgICAgICR7eyB2YXJzLkdFTUlOSV9NT0RFTCB9fQogICAgICAgICAgR09BTF9NQVJBVEhPTl9USU1FOiAgICAgICAgJHt7IHZhcnMuR09BTF9NQVJBVEhPTl9USU1FIH19CiAgICAgICAgICBHT0FMX1JBQ0VfUEFDRTogICAgICAgICAgICAke3sgdmFycy5HT0FMX1JBQ0VfUEFDRSB9fQogICAgICAgICAgUlVOTkVSX1BST0ZJTEU6ICAgICAgICAgICAgJHt7IHZhcnMuUlVOTkVSX1BST0ZJTEUgfX0KICAgICAgICBydW46IHB5dGhvbiBnYXJtaW5fd2Vla2x5X3JlcG9ydC5weQo=',
}

import base64, getpass, json, requests

PAT = getpass.getpass("GitHub Personal Access Token (classic, repo+workflow): ").strip()
REPO = input("作成するリポジトリ名（例: garmin-weekly-report）: ").strip() or "garmin-weekly-report"

H = {"Authorization": f"Bearer {PAT}",
     "Accept": "application/vnd.github+json",
     "X-GitHub-Api-Version": "2022-11-28"}

# ユーザー名を取得
me = requests.get("https://api.github.com/user", headers=H)
me.raise_for_status()
owner = me.json()["login"]
print("GitHubユーザー:", owner)

# リポジトリ作成（既存ならスキップ）
r = requests.post("https://api.github.com/user/repos", headers=H,
                  json={"name": REPO, "private": True,
                        "description": "Garmin weekly running report (Gemini + LINE)"})
if r.status_code == 201:
    print("✅ リポジトリ作成:", r.json()["html_url"])
elif r.status_code == 422:
    print("ℹ️ 同名リポジトリが既にあります。そこに上書きします。")
else:
    r.raise_for_status()

# ファイルを1つずつ Contents API でアップロード
def put_file(path, content_b64):
    url = f"https://api.github.com/repos/{owner}/{REPO}/contents/{path}"
    # 既存ファイルがあれば sha を取得して更新
    sha = None
    ex = requests.get(url, headers=H)
    if ex.status_code == 200:
        sha = ex.json().get("sha")
    body = {"message": f"add {path}", "content": content_b64, "branch": "main"}
    if sha:
        body["sha"] = sha
    resp = requests.put(url, headers=H, json=body)
    resp.raise_for_status()

for path, b64 in PROJECT_FILES_B64.items():
    put_file(path, b64)
    print("  ⬆️", path)

print("\n✅ 全ファイルのアップロード完了")
print("リポジトリ:", f"https://github.com/{owner}/{REPO}")


## 手順4: 秘密情報を登録して自動実行を開始（スマホのブラウザで）

ファイルは配置できました。最後に GitHub 側で秘密情報を登録します。
（安全のため、ここだけは手動で行います）

1. 作成されたリポジトリを開く（手順3の最後に出たURL）
2. **Settings → Secrets and variables → Actions** を開く
3. **Secrets** タブで「New repository secret」から次の4つを登録:

   | Name | Value |
   |------|-------|
   | `GARMIN_TOKENS` | 手順2で生成したトークン文字列（下のセルを実行すると再表示できます） |
   | `GEMINI_API_KEY` | Gemini APIキー |
   | `LINE_CHANNEL_ACCESS_TOKEN` | LINEチャネルアクセストークン |
   | `LINE_USER_ID` | 自分のLINEユーザーID（U始まり） |

4. （任意）**Variables** タブで目標を登録: `GOAL_MARATHON_TIME`, `GOAL_RACE_PACE`, `RUNNER_PROFILE`, `GEMINI_MODEL`
5. **Actions** タブを開き、ワークフローを有効化
6. 「Weekly Running Report」→「**Run workflow**」で手動実行 → LINEに届けば成功 🎉

以降は毎週月曜 7:00（日本時間）に自動でレポートが届きます。


In [ ]:
# GARMIN_TOKENS をもう一度表示（コピーして Secret に貼り付け）
print(GARMIN_TOKENS)
